<a href="https://colab.research.google.com/github/Alt4660/econ5200-ps1/blob/main/Econ_5200_PS1_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Econ 5200 PS1 — The Measurement Audit

**Starter notebook.** The Canvas assignment is the brief: read it first. This
notebook gives you the structure, one heading per step in the order the rubric
grades them, and the module skeleton the brief asks for. Every result is yours
to compute.

**Before you start**
1. *File › Save a copy in Drive*, and rename the copy `Econ 5200 PS1`.
2. Every claim gets a text cell. A deliverable that cannot be read without
   running it is not a deliverable.
3. **AI is not allowed in Phases 1–3.** Phase 4 requires it, with the
   transcript in `ai-appendix.md` in your repository.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)     # one generator, so every draw is reproducible

---
## Phase 1: Reproduce the Wrong Number (15 pts)

### Step 1.1: Build the transaction data

> ### A tool you have not used yet — drawing skewed data
>
> `rng.lognormal(mean, sigma, size)` draws right-skewed positive values: most
> are modest, a few are large. `mean` and `sigma` describe the logarithm of the
> values, not the values themselves, so check `np.median(...)` of a draw
> before you settle on them. `rng.choice(...)` and `rng.integers(...)` are
> useful for the ids and the years.

At least 20,000 rows over two years: `customer_id`, `year`, `basket_value`.
Plant the B2B orders and the change in what gets logged, and record the true
consumer average for each year **before** you contaminate anything.

So I started by simulating two years of consumer transactions, 10,000 orders per year, with basket_value drawn from a lognormal distribution (log-scale mean 4, sigma 0.8). My reason for using a lognormal is because real spending is almost always going to be right skewed since you can't spend negative amounts on an order and therefore there is a “wall / floor” at zero which in this case implies the floor doesn’t apply to the right hand side of the distribution. Therefore most baskets are modest and a few are large. The typical median basket is about 55 dollars, and the mean is about 75 dollars. The mean is going to be higher since it is impacted by extreme values, in this case the long right tail pulls it up.

In [2]:
# Clean consumer transactions

n = 10_000

clean_parts = []

for yr in [1, 2]:
    part = pd.DataFrame({
        "customer_id": rng.integers(1, 5000, n),
        "year": yr,
        "basket_value": rng.lognormal(mean=4, sigma=0.8, size=n),
    })
    clean_parts.append(part)
clean = pd.concat(clean_parts, ignore_index=True)
print(len(clean))
print(np.median(clean["basket_value"]))

20000
54.60554338107531


In [3]:
# The truth: the consumer average per year, recorded before contamination

truth = clean.groupby("year")["basket_value"].mean()

clean["is_b2b"] = False
clean["cancelled"] = False

Contamination 1 explained:

For the first contamination I add a small a number of very large business to business orders that were never meant to be counted as part of a consumer metric. I added 15 in year 1 and 30 in year 2. Their values come from a separate lognormal distribution (log-scale mean 8, sigma 0.5). Which puts the typical (mean) B2B order at about 3,000 dollars, which is roughly 55 times the typical consumer basket of 55 dollars. The averages are about 2,873 dollars in year 1 and 3,281 dollars in year 2 for this contamination .I doubled the number of B2B orders from year 1 to year 2 so that the huge orders have a bigger effect on the dashboard's average in year 2 than in year 1, which is what lets a tiny tail create growth that isn't real. Even so, these orders are well under 1% of all rows. But since the clean orders are all fairly similar in value the mean is very sensitive to extreme values, so a handful of huge orders can still move it noticeably while the typical customer is unchanged. These are extra rows added alongside the consumer data and not replacements for it (otherwise it would defeat the purpose of adding them). As for their format, they use a separate customer ID range (5000 to 5099) and carry the tag is_b2b = True, so I can later repair this fault by itself and measure how much of the gap it causes.

In [4]:
# Contamination 1: B2B orders

n_b2b_y1 = 15
n_b2b_y2 = 30

b2b_parts = []
for yr, k in [(1, n_b2b_y1), (2, n_b2b_y2)]:
    part = pd.DataFrame({
        "customer_id": rng.integers(5000, 5100, k),
        "year": yr,
        "basket_value": rng.lognormal(mean=8, sigma=0.5, size=k),
        "is_b2b": True,
        "cancelled": False,
    })
    b2b_parts.append(part)
b2b = pd.concat(b2b_parts, ignore_index=True)
print(b2b.groupby("year")["basket_value"].agg(["count", "mean"]))

      count         mean
year                    
1        15  2873.105705
2        30  3280.712341


In [5]:
# Contamination 2: what gets logged changes between year 1 and year 2

n_cancel_y2 = 500

cancelled = pd.DataFrame({
    "customer_id": rng.integers(1, 5000, n_cancel_y2),
    "year": 2,
    "basket_value": rng.lognormal(mean=4.3, sigma=0.8, size=n_cancel_y2),
    "is_b2b": False,
    "cancelled": True,
})

Contamination 2 explained:

I add 500 cancelled orders that are logged only in year 2, in year 1 the system did not record cancellations and in year 2 it started to, so the two years no longer measure the same thing. These rows are roughly 5% of year 2's records. Cancelled orders should not be counted as completed sales, so they should not count toward average basket value, but a dashboard that just averages every row has no way to tell and will miscount them. Their values come from a lognormal with a log-scale mean of 4.3 (sigma 0.8), a typical size of about 74 dollars, slightly above the typical completed basket of 55 dollars. That is essentially an assumption I chose. My thought process was that larger baskets are probably more likely to be cancelled. The rows use ordinary customer IDs and carry the tag cancelled = True, so I can later repair this fault on its own and measure its share of the gap.

To explain why this contamination is different from our first; B2B is a tail problem, with a few huge values. This one is a definition problem. What’s happening is many ordinary-sized rows that appear in year 2 and not year 1. That difference will matter when I test which fixes work.

### Step 1.2: Reproduce the dashboard

Compute the naive `mean(basket_value)` by year, as the dashboard does, and
quantify its gap from the truth.

What the dahsboard is showing v.s the truth:

The dashboard computes a plain average of basket_value over every row in each year, with no filtering. On my data the dashboard (naive) average is 79.65 dollars in year 1 and 87.30 dollars in year 2, which is 9.60% growth. The true consumer average is 75.46 dollars in year 1 and 76.65 dollars in year 2, which is 1.58% growth. So the dashboard shows an increase of nearly 10% when real basket values barely moved by more than 1%. That is in the same range as the 8% the client reported. The dashboard overstates the average by 4.19 dollars in year 1 and by 10.65 dollars in year 2. Year 1 contains only the B2B fault, so its 4.19 dollar gap comes entirely from the 15 B2B orders. Year 2 contains both faults, so its larger 10.65 dollar gap includes the B2B orders and the cancelled orders. The gap grew by about 6.46 dollars between the two years, and that growth in the error, not the error itself, is what the dashboard reports as an increase in basket value. Nothing in the dashboard is miscalculated. It computes exactly the formula it was given, on rows that include things it should not. The next phase separates how much of the 6.46 dollars each fault is responsible for.

In [6]:
# The dashboard's number vs the truth

df = pd.concat([clean, b2b, cancelled], ignore_index=True)

naive = df.groupby("year")["basket_value"].mean()

print("truth:", truth.round(2).to_dict())
print("naive:", naive.round(2).to_dict())

true_growth = (truth[2] / truth[1] - 1) * 100
naive_growth = (naive[2] / naive[1] - 1) * 100

print("true growth (%):", round(true_growth, 2))
print("naive growth (%):", round(naive_growth, 2))
print("gap in year 1 ($):", round(naive[1] - truth[1], 2))
print("gap in year 2 ($):", round(naive[2] - truth[2], 2))

truth: {1: 75.46, 2: 76.65}
naive: {1: 79.65, 2: 87.3}
true growth (%): 1.58
naive growth (%): 9.6
gap in year 1 ($): 4.19
gap in year 2 ($): 10.65


---
## Phase 2: Locate the Cause (40 pts)

### Step 2.1: Separate the two mechanisms (25 pts)

Report each contribution as a number.

Separation and impact associated with each contamnation explained:

The dashboard's growth number is too high by 6.46 dollars. The true change in average basket value from year 1 to year 2 is 1.19 dollars, but the dashboard shows 7.65 dollars. So what this step does is it works out how much of that extra 6.46 comes from each of the two contaminations, the B2B orders and the cancelled orders that were still counted.

To separate them I repaired one fault at a time and measured how much fake growth was left. When I removed only the B2B orders, 1.52 dollars of fake growth remained, so the B2B orders accounted for 4.93 dollars. When I removed only the cancelled orders, 5.39 dollars remained, so the cancelled orders accounted for 1.06 dollars. These two pieces add up to 5.99, which is 0.46 short of the 6.46 total. The two faults overlap, so each one looks a little smaller when the other is still in the data. This means the split is approximate, but it doesn't change the conclusion. The B2B orders are the larger problem by a wide margin, about four to five times the size of the cancelled orders problem.

In [7]:
# How much of the gap is the skew and the B2B tail?

true_change = truth[2] - truth[1]
naive_change = naive[2] - naive[1]
total_fake = naive_change - true_change

print("total fake growth ($):", round(total_fake, 2))
# Repair ONLY the B2B fault: keep rows that are not B2B
no_b2b = df[df["is_b2b"] == False]
mean_no_b2b = no_b2b.groupby("year")["basket_value"].mean()

fake_left = (mean_no_b2b[2] - mean_no_b2b[1]) - true_change
b2b_share = total_fake - fake_left
print("fake growth left after removing B2B ($):", round(fake_left, 2))
print("B2B share ($):", round(b2b_share, 2))

total fake growth ($): 6.46
fake growth left after removing B2B ($): 1.52
B2B share ($): 4.93


In [8]:
# How much is the change in what was logged?

no_cancel = df[df["cancelled"] == False]
mean_no_cancel = no_cancel.groupby("year")["basket_value"].mean()

fake_left_2 = (mean_no_cancel[2] - mean_no_cancel[1]) - true_change
logging_share = total_fake - fake_left_2
print("fake growth left after removing cancelled ($):", round(fake_left_2, 2))
print("logging share ($):", round(logging_share, 2))

fake growth left after removing cancelled ($): 5.39
logging share ($): 1.06


| Mechanism | Contribution to the gap | How you isolated it |
|---|---|---|
| Skew and B2B tail | About 4.93 dollars of the 6.46 dollar gap and otherwise roughly 5.39 if repaired first| Removed the B2B orders and measured how much fake growth was left (1.52 dollars) |
| Change in what was logged | About 1.06 dollars (roughly 1.52 if repaired first)|Removed the cancelled orders and measured how much fake growth was left (5.39 dollars) |

### Step 2.2: Robust alternatives (15 pts)

`stats.trim_mean(x, 0.1)` drops the top and bottom 10% before averaging.

So the dashboard says average basket value grew 9.60 percent from year 1 to year 2, but the true growth is only 1.58 percent. In this step I tried three alternatives to the plain average that are less affected by the huge B2B orders: the median, the trimmed mean (which cuts off the top and bottom 10 percent of orders), and the mean after excluding unusually large orders. For the exclusion rule I used a cutoff on the log scale, 1.5 interquartile ranges above the upper quartile, because basket values are skewed and the log makes them roughly symmetric, so the fence is not set too low for normal big orders.

The median gave growth of 5.32 percent, the trimmed mean gave 3.85 percent, and the mean after B2B exclusion gave 3.46 percent. All three are closer to the truth than the dashboard, and the exclusion rule is the closest. My reason for picking it is that it targets the B2B orders directly, since they are far bigger than a normal basket. The trimmed mean also cuts ordinary customers, and the median measures the typical order rather than the average spend, so it answers a slightly different question.

None of them gets all the way down to 1.58 percent though. The reason is that the cancelled orders are normal sized, so there is no rule based on how big an order is that can pick them out. Therefore fixing the cancelled orders is not a matter of choosing a better average, it needs a definition fix, meaning cancelled orders should not be counted as transactions in the first place.

My recommendation is the mean after B2B exclusion, with growth of about 3.46 percent. It assumes that B2B orders are much larger than normal baskets, and it fails if that stops being true, for example if the cutoff is set too loose and B2B orders slip through, or too tight and it cuts real customers. Because of that the cutoff multiplier should be reported next to the number.

In [9]:
# Median, trimmed mean, and the mean after your B2B exclusion rule


def exclude_tail(x, k=1.5):
    """Mean after dropping orders above a Tukey fence computed on the log scale."""
    logs = np.log(x)                                # so I can work on the log scale, where the data will form symmetrical bell
    q1, q3 = logs.quantile([0.25, 0.75])            # quartiles of the logs
    fence = np.exp(q3 + k * (q3 - q1))              # upper fence, converted back to dollars
    return x[x <= fence].mean()                     # average only the orders at or below the fence

grouped = df.groupby("year")["basket_value"]        # the piles, one per year

results = pd.DataFrame({
    "truth": truth,
    "naive mean": grouped.mean(),
    "median": grouped.median(),
    "trimmed mean (10%)": grouped.apply(lambda x: stats.trim_mean(x, 0.1)),
    "mean after B2B exclusion": grouped.apply(exclude_tail),
})
print(results.round(2))

growth = (results.loc[2] / results.loc[1] - 1) * 100
print(growth.round(2))

      truth  naive mean  median  trimmed mean (10%)  mean after B2B exclusion
year                                                                         
1     75.46       79.65   53.76               62.53                     73.37
2     76.65       87.30   56.62               64.94                     75.91
truth                       1.58
naive mean                  9.60
median                      5.32
trimmed mean (10%)          3.85
mean after B2B exclusion    3.46
dtype: float64


| Estimator | Value | The assumption that makes it right | When it would be wrong |
|---|---|---|---|
| Median | 5.32 | The dollar gap between typical orders and big orders isn't too large, so the median is close to the average spend |Fails when bad orders sit near the middle of the distribution, like the cancelled orders, because then the middle itself moves. It also fails in skewed data, because the median measures the typical order and not the average spend. |
| Trimmed mean | 3.85 | 	Contamination is under 10% at each end | Cuts a fixed 10% per end regardless of content, so it removes legitimate large orders and understates average spend; cannot remove contamination above 10% of a tail|
| Mean after B2B exclusion | 3.46%| B2B orders are far larger than normal baskets | Too loose a cutoff leaves B2B orders in; too tight a cutoff cuts large legitimate baskets; cannot catch bad orders that are ordinary in size, like the cancelled orders|

**Which number should the dashboard show?** Commit to one and defend it.

I would show the mean after B2B exclusion, which gives growth of 3.46 percent. The dashboard now shows 9.60 percent and the truth is 1.58 percent, so this is the closest of the three options I tested (the median gave 5.32 percent and the trimmed mean gave 3.85 percent).

I picked it because it deals directly with the biggest cause of the problem, which is the very large B2B orders. The trimmed mean cuts the same amount from every dataset, so it throws away normal big orders too. The median shows the typical order, not the average spend, so it is answering a different question.

It is still not perfect, because 3.46 percent is higher than the true 1.58 percent. The reason is the cancelled orders. They look like normal orders, so no rule based on order size can remove them. To fix that, the dashboard should stop counting cancelled orders as sales.

This answer only works if B2B orders are much bigger than normal ones, and the cutoff I used (1.5) should be shown next to the number, because a different cutoff changes the result.

---
## Phase 3: Ship the Fix (20 pts)

### Step 3.1: The module

The skeleton below fixes the names the brief asks for. Replace each body,
write the docstrings properly, and keep the type hints. Lab 4's
`OutlierDetector` is a model for the shape: one function per job, the method
chosen by a parameter, and no reliance on a global variable.

Phase 3 explained:

For Phase 3 I turned the audit into a small module called basket_metrics.py, so the checks can be reused on any transaction table instead of being buried in the notebook. It has two functions. audit_report takes a table and returns one row for each column, showing its data type, the share of missing values, how skewed it is, and how many outliers it has. For the outliers I used the same rule as before, anything more than 1.5 interquartile ranges beyond the quartiles, and it only does this for numeric columns. robust_mean takes a list of values and returns an average that is less sensitive to extreme values. It has three methods you can choose from, the median, the trimmed mean, and the mean after excluding large orders, and the default is the median.

I wrote a few things into the module on purpose so it is safe to use. Each function has a docstring that says what it does, `robust_mean` and `main` have type hints on the method name and on what they return, and nothing depends on global variables from the notebook. robust_mean drops missing values first, and it raises a ValueError with a clear message if there is nothing left, if you ask for a method that does not exist, or if the exclusion method gets values that are not positive, because it takes logs. I did it this way so that it fails loudly instead of silently returning a wrong number. The last piece is `if __name__ == "__main__": main()`, which means the demo only runs when you execute the file directly and not when you import it.

To test it I wrote assertions on fresh data the module has never seen, generated with a different random seed. They check five things. On symmetric data the median and trimmed mean land close to the plain mean. When I add a few extreme outliers the mean jumps a lot, but the median barely moves. The mean after exclusion comes within 5 percent of the clean mean even with B2B style orders added. A bad method name raises a ValueError. And audit_report returns one row per column, finds the outliers that were planted, and reports the correct share of missing values. All of the assertions passed, which tells me the code does what I claim on data it was not tuned on. It does not tell me the exclusion rule is the right choice for every dataset, that is an assumption which Phase 2 already flagged.

In [10]:
import os
os.makedirs("src", exist_ok=True)

In [11]:
%%writefile src/basket_metrics.py
"""Audit a transaction table and compute a robust average basket value."""

import numpy as np
import pandas as pd
from scipy import stats


def audit_report(df):
    """Check the quality of a table, one row per column. For every column it reports the data type and the share of values that are missing.
    For numeric columns it also reports the skew and the number of outliers (true/false columns don't count as numeric here). An outlier
    is any value more than 1.5 IQRs below the first quartile or above the third quartile, which is Tukey's rule. Text and true/false columns get
    NaN for the skew and outlier count.


    Args: df: Any DataFrame, for example the transaction table.

    Returns: A DataFrame indexed by column name with the columns ``dtype``,
        ``missing_share``, ``skew`` and ``n_outliers``. """
    rows = []
    for col in df.columns:
        s = df[col]
        row = {
            "column": col,
            "dtype": str(s.dtype),
            "missing_share": float(s.isna().mean()),
            "skew": np.nan,
            "n_outliers": np.nan,
        }
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            x = s.dropna()
            q1, q3 = x.quantile([0.25, 0.75])
            iqr = q3 - q1
            row["skew"] = float(x.skew())
            row["n_outliers"] = int(((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).sum())
        rows.append(row)
    return pd.DataFrame(rows).set_index("column")


def robust_mean(x, method: str = "median") -> float:
    """Return a robust centre of ``x`` that resists extreme values.

    Args:
        x: Numbers to summarise. Missing values are dropped.
        Methods:
        One of ``"median"``: the middle value.
        ``"trimmed"``: the mean after dropping the lowest and highest 10% of values.
        ``"exclude"``: the mean after dropping values above an upper fence computed on the log scale, ``exp(Q3 + 1.5 * IQR)`` of the logs.
        This is the B2B exclusion rule. It needs strictly positive values.

    Returns: The robust centre as a float.

    Raises: ValueError: If ``x`` has no valid values, ``method`` is unknown, or
            ``method="exclude"`` is used with non-positive values. """

    values = pd.Series(x, dtype=float).dropna()   # drop missing values first
    if values.empty:
        raise ValueError("x has no valid values")

    if method == "median":
        return float(values.median())
    if method == "trimmed":
        return float(stats.trim_mean(values, 0.1))
    if method == "exclude":
        if (values <= 0).any():                   # logs need positive values
            raise ValueError("method='exclude' needs strictly positive values")
        logs = np.log(values)
        q1, q3 = logs.quantile([0.25, 0.75])
        fence = np.exp(q3 + 1.5 * (q3 - q1))      # upper fence, back in dollars
        return float(values[values <= fence].mean())
    raise ValueError(f"unknown method {method!r}; use 'median', 'trimmed' or 'exclude'")


def main() -> None:
    """Demonstrate both functions on a small simulated, contaminated table."""
    rng = np.random.default_rng(0)
    clean = rng.lognormal(mean=4, sigma=0.8, size=2_000)
    b2b = rng.lognormal(mean=8, sigma=0.5, size=10)
    df = pd.DataFrame({"basket_value": np.concatenate([clean, b2b])})

    print("Audit report:")
    print(audit_report(df).round(2))
    print("\nPlain mean:", round(df["basket_value"].mean(), 2))
    for method in ("median", "trimmed", "exclude"):
        print(f"robust_mean ({method}):", round(robust_mean(df["basket_value"], method), 2))
    print("True clean mean:", round(clean.mean(), 2))


if __name__ == "__main__":
    main()

Writing src/basket_metrics.py


### Step 3.2: Verification checkpoint

Import the module back and test it on data it has not seen. An `assert` stops
with an error when its condition is False, which is the point: a failing
assertion tells you something.

In [12]:
import importlib
import sys
sys.path.insert(0, "src")
import basket_metrics
importlib.reload(basket_metrics)
rng_test = np.random.default_rng(7)

# 1. On symmetric data, the robust centres agree with the ordinary mean
sym = rng_test.normal(loc=100, scale=5, size=5_000)
assert abs(basket_metrics.robust_mean(sym, "median") - sym.mean()) < 0.5
assert abs(basket_metrics.robust_mean(sym, "trimmed") - sym.mean()) < 0.5

# 2. Extreme values wreck the plain mean but barely move the median
base = rng_test.lognormal(mean=4, sigma=0.8, size=5_000)
dirty = np.concatenate([base, np.full(20, 50_000.0)])
assert dirty.mean() - base.mean() > 100
assert abs(basket_metrics.robust_mean(dirty, "median") - basket_metrics.robust_mean(base, "median")) < 1

# 3. The exclusion rule recovers the clean mean to within 5%
assert abs(basket_metrics.robust_mean(dirty, "exclude") - base.mean()) / base.mean() < 0.05

# 4. A bad method name fails loudly instead of returning something wrong
try:
    basket_metrics.robust_mean(base, "nonsense")
    raise AssertionError("expected a ValueError")
except ValueError:
    pass

# 5. audit_report: one row per column, and it sees the planted outliers and missing values
table = pd.DataFrame({"basket_value": dirty, "year": 1})
table.loc[:9, "basket_value"] = np.nan
report = basket_metrics.audit_report(table)
assert len(report) == table.shape[1]
assert report.loc["basket_value", "n_outliers"] >= 20
assert abs(report.loc["basket_value", "missing_share"] - 10 / len(table)) < 1e-9
print("all assertions passed")    # picks up your edits when you re-run %%writefile

all assertions passed


---
## Phase 4: AI Expansion (25 pts)

AI is now allowed and required. The transcript goes in `ai-appendix.md` in your
repository, linked from here: the prompt as sent, the raw reply including what
you rejected, and one line on every change you made.

**Link to `ai-appendix.md`:**

https://github.com/Alt4660/econ5200-ps1/blob/main/Problem%20Set%201/ai-appendix.md




### Task 4.1: The adversarial review (15 pts)

Run the prompt from the brief, then run the strongest objection down.

**The strongest objection, and the evidence that settles it:**

The mean after B2B exclusion only fixes the tail. Cancelled orders appear only in year 2 and are still ordinary sized, so no size based rule could ever actually exclude them. Does the recommendation still overstate growth? Yes and my phase 2 numbers show this already: the exclusion mean gave 3.46% growth against the true number which was 1.58%.

In [13]:
# The analysis that settles it


# 1. Fixed B2B fence from year 1 only, applied to both years
y1_logs = np.log(df.loc[df["year"] == 1, "basket_value"])
q1, q3 = y1_logs.quantile([0.25, 0.75])
fixed_fence = np.exp(q3 + 1.5 * (q3 - q1))

# 2. Drop B2B (by the fence) and cancelled orders in both years
like_for_like = df[(df["basket_value"] <= fixed_fence) & (df["cancelled"] == False)]
lfl_mean = like_for_like.groupby("year")["basket_value"].mean()
lfl_growth = (lfl_mean[2] / lfl_mean[1] - 1) * 100

print("like-for-like means:", lfl_mean.round(2).to_dict())
print("like-for-like growth (%):", round(lfl_growth, 2))
print("true growth (%):", round(true_growth, 2))

# 3. Does the fence catch the planted B2B orders? (answer key check)
flagged = df["basket_value"] > fixed_fence
print(pd.crosstab(df["is_b2b"], flagged, rownames=["actually B2B"], colnames=["above fence"]))

# 4. Bootstrap 95% interval for growth, resampling within each year
def growth_once(data):
    m = data.groupby("year")["basket_value"].mean()
    return (m[2] / m[1] - 1) * 100

boot = []
for _ in range(1000):
    resampled = pd.concat([
        g.sample(len(g), replace=True, random_state=int(rng.integers(1_000_000)))
        for _, g in like_for_like.groupby("year")
    ])
    boot.append(growth_once(resampled))
print("95% interval for growth (%):", np.percentile(boot, [2.5, 97.5]).round(2))

like-for-like means: {1: 73.37, 2: 74.33}
like-for-like growth (%): 1.31
true growth (%): 1.58
above fence   False  True 
actually B2B              
False         20407     93
True              0     45
95% interval for growth (%): [-1.1   3.76]


**What you found, including if it changes your Phase 2 recommendation:**

The objection was correct. on a like-for-like basis growth is 1.31% (average basket 73.37 dollars in year 1, 74.33 dollars in year 2), against a true 1.58%. The Phase 2 exclusion mean gave 3.46%, so it still carried about 2.15 points more growth than the like-for-like version. I made two changes to get from one to the other, removing the cancelled orders and using one fixed cutoff for both years, so I cannot say exactly how the 2.15 points split between them. The cancelled orders are the main suspect, because they are ordinary sized and no size based rule can remove them. As for the fence: all 45 B2B orders were above the fence and none slipped through. 93 non-B2B rows were also above it. These include cancelled orders, but most are legitimate large baskets which we mustn't dismiss. That is why the like-for-like levels sit about 2 dollars below the truth. The rule is a good estimator of growth but it's still a biased estimator of level, so it's not safe to reconcile it as revenue level. Uncertainty in our analysis is still very much present. The bootstrap 95% interval for growth is -1.10% to 3.76%. It contains the true 1.58%, but it also contains zero, so I cannot claim growth is positive. The interval ignores uncertainty in choosing the fence, and this is one simulation. This has changed my Phase 2 recommendation. I previously recommended the mean after B2B exclusion alone. Now I would recommend the B2B exclusion rule together with excluding cancelled orders in both years, reported with its cutoff multiplier and its interval.

### Task 4.2: The board slide (10 pts)

150 words at most: the headline number, the correction, the confidence, and the
one thing you would still want before betting on it.

Average basket value grew about 1.3%, not the 9.6% the dashboard reports.

Two faults inflated it: a handful of very large B2B orders (about 4.93 dollars of the 6.46 dollar overstatement) and cancelled orders logged only in year 2 (about 1.06 dollars). Excluding B2B orders with one fixed cutoff and removing cancelled orders in both years gives 1.31% growth, against a true 1.58% in my simulation.

Confidence is moderate. The 95% interval runs from -1.10% to 3.76%, so it contains the truth but also zero, which means I cannot claim growth is positive. The filtered average sits about 2 dollars below the true level, so it measures growth well but should not be reconciled to revenue.

Before betting on it, I would confirm the client's real data has a reliable cancellation field, and how year 1 handled cancellations.

---
### Before you submit
- [ ] Truth recorded before contamination, and both mechanisms reported as numbers
- [ ] `src/basket_metrics.py` pushed with the notebook, and at least three assertions passing
- [ ] `ai-appendix.md` in the repository and linked above
- [ ] *Runtime › Restart and run all* runs without errors